# 02 Certify links (blind)

**Task.** From the data alone, report which farm follows which, with $\Pr(\text{any false link})\le\alpha$.

**Reads:** `data/measurements/`, `data/sketch/`. **Never reads:** `data/truth/`.

**How to read the tags.** A code cell that uses a result of the paper starts with a comment such as
`# [Thm 2] threshold c_alpha`. The tag names the result; the rest of the line says what the cell does. Numbering
follows the paper: Definition 1 and Lemma 1 (first lags), Theorem 2 (certified links), Proposition 3 (lag scan),
Corollary 4 (channel certificates), Algorithm 1 and the Procedure of Section 3, equations (3) (impulse response),
(4) (residuals), (5) (the statistic), (6) (the threshold), (7) (power) and (8) (measurement duration), and the
conditions C1 (measured innovations) and C2 (predictable regressors and weights) of Theorem 2.

In [1]:
import os, sys, glob, pickle, warnings, numpy as np, pandas as pd
warnings.filterwarnings('ignore')            # scikit-learn convergence messages from the Lasso baseline
sys.path.insert(0, 'lib')
import certify as K, farmlib as FL
np.set_printoptions(precision=2, suppress=True, linewidth=150)
files = sorted(glob.glob('data/measurements/*.npz'))
SK = FL.load_sketch('data/sketch/channel_sketch.npz')
TAU, TAU0 = FL.lags(SK), FL.tau0(SK)          # lags tau_{d,o} and tau^0_c, from the sketch alone
ALPHA = 0.1
os.makedirs('results', exist_ok=True)
print(len(files), 'communities; reading only data/measurements and data/sketch')

10 communities; reading only data/measurements and data/sketch


## Where to look: lags from the channel sketch
The sketch says only which channel moves which. Definition 1 and Lemma 1 give the first lag $\tau^W$ at which farm
$j$'s excitation $d$ can reach channel $o$ of another farm through one social crossing. The pair $(d,o)$ is tested
at $\tau_{d,o}=\tau^W$ only if the social network arrives first: before the physical network and before any chain
with two or more crossings ($\tau_2$).

In [2]:
# [Def 1, Lemma 1] first lags from the zero pattern alone
_, detail = FL.lags(SK, detail=True)
pd.DataFrame([dict(excitation=FL.EXCITATIONS[d], output=FL.NAMES[o], tau_W=a['W'], tau_2=a['tau2'],
                   tested_lag=TAU.get((d, o), 'not reached')) for (d, o), a in detail.items()])

,excitation,output,tau_W,tau_2,tested_lag
0,canal-turn deviation,soil moisture,4.0,5.0,4
1,canal-turn deviation,GW level,4.0,5.0,4
2,canal-turn deviation,belief survey,3.0,4.0,3
3,canal-turn deviation,pumping,3.0,4.0,3
4,canal-turn deviation,spraying,NaN,NaN,not reached
5,canal-turn deviation,canopy temp. (decoy),NaN,NaN,not reached
6,canal-turn deviation,price survey (decoy),NaN,NaN,not reached
7,pest-pressure innovation,soil moisture,NaN,NaN,not reached
8,pest-pressure innovation,GW level,NaN,NaN,not reached
9,pest-pressure innovation,belief survey,NaN,NaN,not reached


Canal $\to$ pumping and perceived scarcity at lag 3; canal $\to$ soil moisture and groundwater level at lag 4;
pest $\to$ spraying at lag 2; nothing reaches the decoys. These are the paper's predictions *before any data*
(Section 4).

## Step 1: residuals that use only the past (eq. 4, condition C2)
For each receiver $i$ and output $o$, the predictable part is removed by a ridge regression fitted on earlier rows
only, refitted weekly after a 90-day warm-up. Row $k$ of the regressors $\mathbf X_{i,o}$ holds:
* a constant;
* $i$'s own daily outputs at the two most recent lags allowed: output $o$ itself at lags 1, 2, and another channel
  $c$ only before $k-\tau+\tau^0_c$;
* $i$'s own excitations and canal schedule at lags $1,\dots,\tau+1$;
* the shared series at lags $0,\dots,3$;
* the other farms' daily outputs at lag $\tau$ and their excitations at lag $\tau+1$.

None of these can depend on the sender's innovation on day $k-\tau$ (Appendix B, Step 1 satisfies C2).

In [3]:
PLAN = 'Pump+Spray'; YEARS = 3; T = YEARS * 365
pl = FL.plan(PLAN)
R0 = FL.load_measurements(files[0])
Rv = FL.view(R0, T, pl['outputs'], pl['excitations'], pl['shared'], pl.get('farms_out'))

# [Step 1, eq. (4), C2] online residual of farm 1's spraying, tested at the pest -> spraying lag
d, o = 1, 'spr'; tau = TAU[(d, o)]; i = 0
t0_d = {c: v for (dd, c), v in TAU0.items() if dd == d}
res = K.residuals(Rv, o, i, tau, pl['excitations'], pl['shared'], tau0=t0_d)
raw = Rv['Y'][o][:, i]
print(f'lag tau={tau};  s.d. of spraying {raw[200:].std():.3f}  ->  s.d. of its residual {np.nanstd(res[200:]):.3f}')

lag tau=2;  s.d. of spraying 1.077  ->  s.d. of its residual 0.211


## Steps 2-3: the statistic, by hand for one pair
For sender $j$ and receiver $i$, on $j$'s event days $k$:
$$D_k=\bar{\mathbf M}_d(k,j)\,\bar{\mathbf R}_o(k,i)=\frac{m_{j,d}[k]}{\hat\sigma_{m,j}[k]}\cdot\frac{\tilde y_{i,o}[k+\tau]}{\hat\sigma_{\tilde y,i,o}[k]},\qquad
\mathbf A_{ij}=\sum_kD_k,\qquad \mathbf V^2_{ij}=\sum_kD_k^2,\qquad \mathbf Z_{ij}=\mathbf A_{ij}/\sqrt{\mathbf V^2_{ij}}.$$
The scales $\hat\sigma$ are root-mean-squares over the days **before** $k$, which keeps C2. The cell recomputes
$Z$ in a few lines and checks it against the library.

In [4]:
# [Steps 2-3, eq. (5)] Z for one pair, by hand, then compared with certify.test_excitation
j, i = 1, 0                                           # sender farm 2 -> receiver farm 1
Tn = Rv['Y'][o].shape[0]; ks = np.arange(90 + 14, Tn - tau)
res = K.residuals(Rv, o, i, tau, pl['excitations'], pl['shared'], tau0=t0_d)
f2 = np.where(np.isnan(res), 0, res)**2
s_y = np.sqrt(np.maximum(np.cumsum(f2)[ks - 1], 1e-12) / np.maximum(np.cumsum(~np.isnan(res))[ks - 1], 1))  # past only
yt = np.where(np.isnan(res[ks + tau]), 0, res[ks + tau])
m = Rv['M'][d][:, j]; ev = Rv['Pi'][d][:, j]
s_m = np.sqrt(np.maximum(np.cumsum(m**2)[ks - 1], 1e-12) / np.maximum(np.cumsum(ev)[ks - 1], 1))              # past only
Dk = (m[ks] / s_m) * (yt / s_y) * ev[ks]
A, V2 = Dk.sum(), (Dk**2).sum()
Z_hand = A / np.sqrt(V2)
Z_lib, _ = K.test_excitation(Rv, d, {o: tau}, ALPHA, pl['excitations'], pl['shared'], tau0=TAU0)
print(f'events n_j={int(ev[ks].sum())}  A={A:.2f}  V^2={V2:.1f}  Z(hand)={Z_hand:.4f}  Z(library)={Z_lib[i, j]:.4f}')
assert abs(Z_hand - Z_lib[i, j]) < 1e-9

events n_j=989  A=197.76  V^2=508.4  Z(hand)=8.7707  Z(library)=8.7707


## Step 3: all pairs at once (the matrix form of eq. 5)
With data matrices (days $\times$ farms), the whole test for one excitation and one output is two matrix products:
$$\mathbf A=\bar{\mathbf R}_o^\top(\bm\Pi_d\odot\bar{\mathbf M}_d),\qquad
\mathbf V^2=(\bar{\mathbf R}_o^{\odot2})^\top(\bm\Pi_d\odot\bar{\mathbf M}_d)^{\odot2},\qquad
\mathbf Z=\mathbf A\oslash\sqrt{\mathbf V^2},$$
where $\bar{\mathbf R}_o(k,i)$ is receiver $i$'s residual on day $k+\tau$ and $\bar{\mathbf M}_d(k,j)$ is sender $j$'s
excitation, each divided by its root-mean-square over earlier days. Rows of $\mathbf Z$ are receivers and columns
are senders.

In [5]:
# [Step 3, eq. (5), matrix form] Z for all 56 pairs from two matrix products, checked against the library
N = Rv['Y'][o].shape[1]
Rbar = np.zeros((len(ks), N)); Mbar = np.zeros((len(ks), N)); Pi = Rv['Pi'][d][ks].astype(float)
for ii in range(N):                                    # receivers: residual on day k+tau, scaled by its past RMS
    r_i = K.residuals(Rv, o, ii, tau, pl['excitations'], pl['shared'], tau0=t0_d)
    f2 = np.where(np.isnan(r_i), 0, r_i)**2; cnt = np.cumsum(~np.isnan(r_i))
    s_y = np.sqrt(np.maximum(np.cumsum(f2)[ks - 1], 1e-12) / np.maximum(cnt[ks - 1], 1))
    Rbar[:, ii] = np.where(np.isnan(r_i[ks + tau]), 0, r_i[ks + tau]) / s_y
for jj in range(N):                                    # senders: excitation scaled by its past RMS
    m_j = Rv['M'][d][:, jj]; ev_j = Rv['Pi'][d][:, jj]
    s_m = np.sqrt(np.maximum(np.cumsum(m_j**2)[ks - 1], 1e-12) / np.maximum(np.cumsum(ev_j)[ks - 1], 1))
    Mbar[:, jj] = m_j[ks] / s_m
A_mat = Rbar.T @ (Pi * Mbar)
V2_mat = (Rbar**2).T @ (Pi * Mbar)**2
Z_mat = A_mat / np.sqrt(V2_mat); np.fill_diagonal(Z_mat, np.nan)
print('largest difference between the matrix form and the library over all pairs:', float(np.nanmax(np.abs(Z_mat - Z_lib))))
print('Z (rows = receivers, columns = senders):'); print(np.round(Z_mat, 1))

largest difference between the matrix form and the library over all pairs: 1.9539925233402755e-14
Z (rows = receivers, columns = senders):
[[ nan  8.8 -1.6  1.7  7.2 -2.7 -0.4  7.4]
 [10.1  nan  8.1 -0.2 -0.   0.8  1.6 -0.1]
 [-1.3  8.9  nan  2.2 -1.7  2.2 -0.1  6.4]
 [ 8.6  8.   5.2  nan -1.5  1.8 -0.1 -0.6]
 [-0.3  0.3  5.7  7.8  nan -0.1  6.5 -0.6]
 [ 0.2 -0.5 12.2  1.2 -0.8  nan  4.4 -1.5]
 [-0.  -0.4  1.6 -0.9 -1.  12.3  nan  6.6]
 [ 7.9  0.2  7.1 -0.4  0.9  0.9  4.9  nan]]


## The threshold and the decisions (Theorem 2)
With $n_p=N(N-1)=56$ ordered pairs, the level is split over the plan's excitations ($\alpha/|\mathcal D|$ each). The
threshold is
$$c_\alpha=\sqrt{2\kappa\log(2n_pn_\kappa/\alpha)},\qquad n_\kappa=\lceil\log_\kappa(b_{\rm hi}/b_{\rm lo})\rceil,$$
with $[b_{\rm lo},b_{\rm hi}]=[n_j/100,\,100\,n_j]$ and $\kappa$ chosen to make $c_\alpha$ smallest. A pair is
reported if $|\mathbf Z_{ij}|\ge c_\alpha$ and $\mathbf V^2_{ij}\in[b_{\rm lo},b_{\rm hi}]$.

In [6]:
# [Thm 2, eq. (6)] threshold c_alpha
n_p = N * (N - 1); alpha_d = ALPHA / len(pl['excitations'])
c, kappa, n_kappa = K.threshold(n_p, alpha_d)
print(f'n_p={n_p}, alpha per excitation={alpha_d}:  kappa={kappa:.2f}, n_kappa={n_kappa}, c_alpha={c:.3f}')
print('recomputed from the formula:', round(float(np.sqrt(2 * kappa * np.log(2 * n_p * n_kappa / alpha_d))), 3))

n_p=56, alpha per excitation=0.05:  kappa=1.08, n_kappa=120, c_alpha=5.197
recomputed from the formula: 5.197


In [7]:
# [Thm 2, Algorithm 1] certify every community for several plans and durations, with adjusted p-values
RUN = [('Pump+Spray', y) for y in (1, 3, 5)] + [('All practical', 3), ('All practical', 5), ('Spray+Pest', 3)]
out = []
for f in files:
    name = os.path.splitext(os.path.basename(f))[0]; R = FL.load_measurements(f)
    for plan_name, yrs in RUN:
        p_ = FL.plan(plan_name)
        Rv_ = FL.view(R, yrs * 365, p_['outputs'], p_['excitations'], p_['shared'], p_.get('farms_out'))
        W_hat, Zs, infos = K.certify_plan(Rv_, p_, TAU, ALPHA, tau0=TAU0)
        out.append(dict(community=name, plan=plan_name, years=yrs, W_hat=W_hat, Z=Zs,
                        p=K.p_values(Zs, infos, ALPHA), info=infos))
pickle.dump(out, open('results/links.pkl', 'wb'))
pd.DataFrame([dict(community=r['community'], plan=r['plan'], years=r['years'], links_reported=int(r['W_hat'].sum()))
              for r in out]).pivot_table(index='community', columns=['plan', 'years'], values='links_reported')

plan         All practical       Pump+Spray             Spray+Pest
years                    3     5          1     3     5          3
community                                                         
community_00          17.0  20.0        3.0  17.0  20.0       18.0
community_01          18.0  20.0        4.0  18.0  20.0       19.0
community_02          18.0  22.0        0.0  18.0  22.0       20.0
community_03          16.0  19.0        3.0  16.0  19.0       16.0
community_04          18.0  20.0        3.0  18.0  20.0       18.0
community_05          18.0  20.0        2.0  20.0  20.0       20.0
community_06          13.0  17.0        3.0  13.0  18.0       15.0
community_07          16.0  20.0        1.0  16.0  20.0       18.0
community_08          17.0  20.0        2.0  17.0  20.0       19.0
community_09          16.0  20.0        2.0  17.0  20.0       17.0

The table shows **how many** links each run reported. Whether they are right is unknown here; that is
scored in notebook 04, the only notebook that opens the truth.

In [8]:
# the reported links of one community, with family-wise adjusted p-values (blind)
r = [x for x in out if x['community'] == 'community_00' and x['plan'] == 'Pump+Spray' and x['years'] == 3][0]
rep = [(f'{j+1}->{i+1}', f"{float(r['p'][i, j]):.1e}") for i in range(N) for j in range(N) if r['W_hat'][i, j]]
print(len(rep), 'reported links (sender->receiver, adjusted p-value):'); print(rep)
assert all(float(p) <= ALPHA for _, p in rep)

17 reported links (sender->receiver, adjusted p-value):
[('2->1', '9.2e-12'), ('5->1', '8.7e-07'), ('8->1', '2.7e-07'), ('1->2', '1.0e-16'), ('3->2', '2.2e-09'), ('2->3', '3.4e-12'), ('8->3', '1.9e-04'), ('1->4', '3.0e-11'), ('2->4', '4.8e-09'), ('3->5', '7.6e-03'), ('4->5', '2.1e-08'), ('7->5', '9.9e-05'), ('3->6', '2.0e-26'), ('6->7', '9.6e-27'), ('8->7', '6.1e-05'), ('1->8', '9.0e-09'), ('3->8', '2.5e-06')]


## Without a sketch: the lag scan (Proposition 3)
Scan lags $\tau=1,\dots,\bar L=6$ for each (excitation, output) pair at level $\alpha/(\bar L|\mathcal D||\mathcal O|)$,
and stop at the first lag with a report. Lags below the true lag $\tau^\ast$ are safe; the only risk is passing
$\tau^\ast$ without a report.

In [9]:
# [Prop 3] lag scan, ignoring the sketch
scan = []
for f in files:
    name = os.path.splitext(os.path.basename(f))[0]; R = FL.load_measurements(f)
    for yrs in (3, 5):
        p_ = FL.plan('Pump+Spray')
        Rv_ = FL.view(R, yrs * 365, p_['outputs'], p_['excitations'], p_['shared'])
        W_hat, found = K.lag_scan(Rv_, p_, ALPHA)
        scan.append(dict(community=name, years=yrs, W_hat=W_hat, found=found))
pickle.dump(scan, open('results/scan.pkl', 'wb'))
pd.DataFrame([dict(community=s['community'], years=s['years'], links_reported=int(s['W_hat'].sum()),
                   **{f"{['canal', 'pest'][d]}->{FL.NAMES[o]}": ('-' if s['found'][(d, o)] is None else s['found'][(d, o)])
                      for (d, o) in s['found']}) for s in scan])

,community,years,links_reported,canal->pumping,canal->spraying,pest->pumping,pest->spraying
0,community_00,3,17,-,-,-,2
1,community_00,5,20,3,-,-,2
2,community_01,3,18,-,-,-,2
3,community_01,5,20,3,-,-,2
4,community_02,3,17,-,-,-,2
5,community_02,5,22,-,-,-,2
6,community_03,3,14,-,-,-,2
7,community_03,5,19,3,-,-,2
8,community_04,3,13,-,-,-,2
9,community_04,5,19,3,-,-,2


Where the scan finds something, it stops at the sketch's lag: lag 2 for pest $\to$ spraying and, with enough
data, lag 3 for canal $\to$ pumping. `-` means nothing was reported at any lag yet: inconclusive, never wrong.